# 02. Exploratory Data Analysis & Feature Engineering
## VLSI & Hardware Fault Diagnosis

In this notebook, we perform comprehensive EDA, investigate physical parameters across fault classes, and build domain-specific engineered features.


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_raw_data, get_feature_target_split, FAULT_CLASS_MAP, BASE_NUMERICAL_FEATURES
from preprocessing import VLSIFeatureEngineer, build_preprocessor

df = load_raw_data("../Data/vlsi_fault_detection.csv")
X, y, meta = get_feature_target_split(df)
print("Features shape:", X.shape)


### 1. Feature Distributions Across Fault Classes
We inspect the mean and standard deviation of each electrical measurement across all six fault classes.


In [ ]:
means = df.groupby('fault_type')[BASE_NUMERICAL_FEATURES].mean().T
means.columns = [FAULT_CLASS_MAP[i] for i in range(6)]
means


### 2. Physical Outlier Justification
Notice that junction temperatures reach up to 120°C and signal integrity margins fall below 0. 
In classical statistics, these may appear as extreme outliers beyond $Q_3 + 1.5 	imes 	ext{IQR}$.
However, in semiconductor physics, extreme values are the **exact signatures** of Thermal and Signal Integrity faults.
Therefore, removing them would destroy the ground-truth signals.


In [ ]:
# Check IQR outliers for key physical parameters
for col in ['junction_temperature', 'total_power', 'si_margin']:
    q25, q75 = np.percentile(df[col], [25, 75])
    iqr = q75 - q25
    lower, upper = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col:22s} | IQR Outliers: {len(outliers):5d} | Fault Type Distribution:\n{outliers['fault_type'].value_counts().to_dict()}")


### 3. Domain-Specific Feature Engineering
We construct 7 physics-based derived features:
1. $V^2 \cdot f$: Dynamic power scaling indicator.
2. $V / f$: Timing slack pressure index.
3. $\Delta T = T_j - T_{\text{ambient}}$: Thermal gradient across the silicon package.
4. Thermal resistance proxy: $\Delta T / P_{\text{total}}$.
5. Dynamic switching stress: $\alpha \cdot C_L \cdot V^2 \cdot f$.
6. Signal integrity health ratio: $\text{SI Margin} / (\text{Noise Margin} + \epsilon)$.
7. Aging thermal compound stress: $\text{Aging Factor} \times (T_j / 100)$.


In [ ]:
fe = VLSIFeatureEngineer()
X_eng = fe.fit_transform(X)
print("Features after engineering:", X_eng.shape)
X_eng[fe.engineered_feature_names_].head()


### 4. Leak-Free Preprocessing Pipeline
We verify that the `ColumnTransformer` (StandardScaler on numerical features and OneHotEncoder on `tech_node`) transforms the feature space cleanly.


In [ ]:
all_num = BASE_NUMERICAL_FEATURES + fe.engineered_feature_names_
preprocessor = build_preprocessor(all_num, ['tech_node'])
X_trans = preprocessor.fit_transform(X_eng)
print("Transformed feature matrix shape:", X_trans.shape)
